# Multi-Sequence / Multi-Class Evaluation — Combined P3-6 vs PGD-mask

**Goal:** Run Combined P3-6 **and** PGD-mask on 2–3 UA-DETRAC sequences with bus/van classes,
70 frames per sequence. Results feed **§4.18** of the PFE report.

| Cell | Content | GPU? | Time (T4) |
|------|---------|------|-----------|
| M1 | Clone / pull repo | No | 1 min |
| M2 | Install dependencies | No | 3 min |
| M3 | Mount Drive + copy weights + images | No | 5 min |
| M4 | Scan sequences for bus/van classes | No | 2 min |
| M5 | Sanity check (5 frames, latent only) | Yes | ~15 min |
| M6 | **Combined P3-6** — 70 frames × sequences | Yes | ~2–2.5h/seq |
| M6b | **PGD-mask baseline** — same frames | Yes | ~20 min/seq |
| M7 | Per-sequence comparison table | No | 1 min |
| M8 | Save everything to Drive | No | 2 min |

**Total estimated time (2 sequences):** ~5–6h on T4.  
Every cell is **resumable** — re-run after disconnect, already-processed frames are skipped.

## Cell M1 — Clone / pull repo

In [ ]:
import os
REPO_DIR = '/content/pfe-latent-attack'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/03aziz03/pfe-latent-attack.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!git log --oneline -5

## Cell M2 — Install dependencies

In [ ]:
%pip install -q "ultralytics>=8.2,<9" lpips>=0.1.4 "torchmetrics>=1.0" diffusers transformers accelerate
import os; os.environ['WANDB_DISABLED'] = 'true'
print('Dependencies installed.')

## Cell M3 — Mount Drive + copy weights + sequence images

**Drive layout expected:**
```
MyDrive/PFE/
  runs/yolov8n_detrac/best.pt
  DETRAC/
    Insight-MVT_Annotation_Train/
      MVI_20011/  MVI_20032/  ...    ← sequence image folders
    DETRAC-Train-Annotations-XML/
      MVI_20011.xml  MVI_20032.xml  ...
```
Adjust `DRIVE_ROOT` if your layout differs.

In [ ]:
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/PFE'
DETRAC_IMGS = f'{DRIVE_ROOT}/DETRAC/Insight-MVT_Annotation_Train'
DETRAC_XML  = f'{DRIVE_ROOT}/DETRAC/DETRAC-Train-Annotations-XML'
WEIGHTS_SRC = f'{DRIVE_ROOT}/runs/yolov8n_detrac/best.pt'

os.makedirs('runs/yolov8n_detrac', exist_ok=True)
if not os.path.exists('runs/yolov8n_detrac/best.pt'):
    shutil.copy(WEIGHTS_SRC, 'runs/yolov8n_detrac/best.pt')
    print('Weights copied.')
else:
    print('Weights already present.')

seqs = sorted(os.listdir(DETRAC_IMGS)) if os.path.exists(DETRAC_IMGS) else []
xmls = sorted(os.listdir(DETRAC_XML))  if os.path.exists(DETRAC_XML)  else []
print(f'Sequences in Drive: {len(seqs)}')
print(f'XML annotations:    {len(xmls)}')
print('First 10:', seqs[:10])

## Cell M4 — Scan sequences for bus / van classes

Ranks all sequences by bus+van instance count. Auto-selects top 3 with images available.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

XML_DIR  = Path(DETRAC_XML)
IMGS_DIR = Path(DETRAC_IMGS)
CLASS_MAP = {'car': 0, 'bus': 1, 'van': 2, 'others': 3, 'other': 3}

seq_stats = {}
for xml_path in sorted(XML_DIR.glob('*.xml')):
    seq_name = xml_path.stem
    counts = defaultdict(int)
    try:
        root_el = ET.parse(xml_path).getroot()
        n_frames = 0
        for frame_el in root_el.findall('.//frame'):
            n_frames += 1
            for target in frame_el.findall('.//target'):
                attr = target.find('attribute')
                if attr is not None:
                    vtype = attr.get('vehicle_type', 'others').lower()
                    counts[CLASS_MAP.get(vtype, 3)] += 1
        seq_stats[seq_name] = {
            'car': counts[0], 'bus': counts[1],
            'van': counts[2], 'others': counts[3],
            'n_frames': n_frames,
            'has_imgs': (IMGS_DIR / seq_name).exists(),
        }
    except Exception as e:
        print(f'  [WARN] {seq_name}: {e}')

ranked = sorted(seq_stats.items(), key=lambda x: x[1]['bus'] + x[1]['van'], reverse=True)

print(f'\n{"Sequence":<16} {"Frames":>7} {"Car":>7} {"Bus":>7} {"Van":>7} {"Others":>7}  imgs?')
print('-' * 68)
for seq, s in ranked[:20]:
    marker = '  ← GOOD' if s['bus'] + s['van'] > 200 and s['has_imgs'] else ''
    print(f"{seq:<16} {s['n_frames']:>7} {s['car']:>7} {s['bus']:>7} {s['van']:>7} "
          f"{s['others']:>7}  {'v' if s['has_imgs'] else 'x'}{marker}")

SELECTED = [
    seq for seq, s in ranked
    if s['has_imgs'] and (s['bus'] + s['van']) > 0
][:3]

print(f'\n>>> Auto-selected: {SELECTED}')
print('Override: edit SELECTED in the next cell if needed.')

In [ ]:
# ── OPTIONAL: override sequence selection ───────────────────────────────────
# SELECTED = ['MVI_20032', 'MVI_20034', 'MVI_40141']

import shutil
from pathlib import Path

N_FRAMES_PER_SEQ = 70

for seq in SELECTED:
    src_dir = Path(DETRAC_IMGS) / seq
    dst_dir = Path(f'data/multiclass/{seq}')
    dst_dir.mkdir(parents=True, exist_ok=True)

    all_imgs = sorted(src_dir.glob('*.jpg'))
    stride   = max(1, len(all_imgs) // N_FRAMES_PER_SEQ)
    selected_imgs = all_imgs[::stride][:N_FRAMES_PER_SEQ]

    copied = sum(1 for img in selected_imgs
                 if not (dst_dir / img.name).exists()
                 and shutil.copy(img, dst_dir / img.name) is not None or
                 (dst_dir / img.name).exists())
    print(f'{seq}: {len(selected_imgs)} frames ready in data/multiclass/{seq}/')

print('\nFrame copy done — ready for M5.')

## Cell M5 — Sanity check (5 frames, latent attack only)

Verifies pipeline on 5 frames before the full run. Expected: DFR > 0, no crash.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

SEQ_SANITY = SELECTED[0]
!python scripts/run_phase3_ablation.py \
    --configs configs/phase3_combined.yaml \
    --data    data/multiclass/{SEQ_SANITY} \
    --n_frames 5 \
    --output  results/multiclass_sanity \
    --resume

In [ ]:
import json
s = json.load(open('results/multiclass_sanity/phase3_combined/summary.json'))
print('=== Sanity (5 frames) ===')
print(f"  DFR_prop  : {s['mean_dfr']:+.1%}")
print(f"  ASR       : {s['mean_asr']:.1%}")
print(f"  LPIPS     : {s['mean_lpips']:.3f}")
ok = s['mean_dfr'] > 0
print('\nPipeline OK — proceed to M6.' if ok else 'WARNING: DFR=0 — check weights path.')

## Cell M6 — Combined P3-6 — 70 frames × all selected sequences

Resumable. Re-run after disconnect — done frames are skipped.  
~2–2.5 h per sequence on T4.

In [ ]:
import os, time
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

for seq in SELECTED:
    print(f'\n{"="*55}\n  COMBINED P3-6: {seq}\n{"="*55}')
    t0 = time.time()
    !python scripts/run_phase3_ablation.py \
        --configs configs/phase3_combined.yaml \
        --data    data/multiclass/{seq} \
        --n_frames {N_FRAMES_PER_SEQ} \
        --output  results/multiclass/{seq} \
        --resume \
        --save_images
    print(f'  Done in {(time.time()-t0)/60:.1f} min')

## Cell M6b — PGD-mask baseline — same frames × all selected sequences

Runs `scripts/run_pgd_baseline.py` — same metric pipeline as the latent attack.  
Same `eps=8/255`, `steps=50`, `mask_restricted=True` as Phase 1.5.  
~15–20 min per sequence on T4 (much faster than Combined).

In [ ]:
import os, time
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

for seq in SELECTED:
    print(f'\n{"="*55}\n  PGD-MASK: {seq}\n{"="*55}')
    t0 = time.time()
    !python scripts/run_pgd_baseline.py \
        --data      data/multiclass/{seq} \
        --n_frames  {N_FRAMES_PER_SEQ} \
        --output    results/multiclass/{seq} \
        --eps       0.03137 \
        --alpha     0.00392 \
        --steps     50 \
        --resume
    print(f'  Done in {(time.time()-t0)/60:.1f} min')

## Cell M7 — Per-sequence comparison table: Combined vs PGD-mask

Generates the full table for **§4.18** with both methods side by side.

In [ ]:
import json, math, numpy as np
from pathlib import Path
from collections import defaultdict
import xml.etree.ElementTree as ET

def load_summary(path):
    if Path(path).exists():
        return json.load(open(path))
    return None

def fmt_pct(v, se=None):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return 'N/A'
    s = f'{v:+.1%}'
    if se is not None:
        s += f'\u00b1{se:.1%}'
    return s

rows_combined = []
rows_pgd      = []

for seq in SELECTED:
    # class distribution from XML
    xml_path = Path(DETRAC_XML) / f'{seq}.xml'
    cls_counts = defaultdict(int)
    if xml_path.exists():
        for t in ET.parse(xml_path).getroot().findall('.//target'):
            a = t.find('attribute')
            if a is not None:
                vt = a.get('vehicle_type', 'others').lower()
                cls_counts[{'car':'car','bus':'bus','van':'van'}.get(vt,'others')] += 1
    cls_str = ', '.join(f"{k}({v})" for k,v in sorted(cls_counts.items()) if v>0)

    s_comb = load_summary(f'results/multiclass/{seq}/phase3_combined/summary.json')
    s_pgd  = load_summary(f'results/multiclass/{seq}/pgd_mask/summary.json')

    if s_comb:
        rows_combined.append({'seq': seq, 'cls': cls_str, **s_comb})
    if s_pgd:
        rows_pgd.append({'seq': seq, 'cls': cls_str, **s_pgd})

# ── Print comparison ──────────────────────────────────────────────────────────
HDR = f'{"Sequence":<14} {"N":>4}  {"Classes":<26}  {"DFR_prop":>12}  {"DFR_str":>8}  {"ASR":>6}  {"LPIPS":>7}'
SEP = '-' * 85

print('\n>>> COMBINED P3-6')
print(HDR); print(SEP)
dfr_c, str_c, asr_c, lpips_c = [], [], [], []
for r in rows_combined:
    print(f"{r['seq']:<14} {r['n_frames']:>4}  {r['cls']:<26}  "
          f"{fmt_pct(r['mean_dfr'], r['se_dfr']):>12}  "
          f"{r['dfr_strict_rate']:>8.1%}  {r['mean_asr']:>6.1%}  {r['mean_lpips']:>7.3f}")
    dfr_c.append(r['mean_dfr']); str_c.append(r['dfr_strict_rate'])
    asr_c.append(r['mean_asr']); lpips_c.append(r['mean_lpips'])
if rows_combined:
    print(SEP)
    n_tot = sum(r['n_frames'] for r in rows_combined)
    print(f"{'AGGREGATE':<14} {n_tot:>4}  {'':26}  "
          f"{np.mean(dfr_c):>+11.1%}   {np.mean(str_c):>8.1%}  {np.mean(asr_c):>6.1%}  {np.mean(lpips_c):>7.3f}")

print('\n>>> PGD-MASK BASELINE (eps=8/255, 50 steps, mask_restricted=True)')
print(HDR); print(SEP)
dfr_p, str_p, asr_p, lpips_p = [], [], [], []
for r in rows_pgd:
    print(f"{r['seq']:<14} {r['n_frames']:>4}  {r['cls']:<26}  "
          f"{fmt_pct(r['mean_dfr'], r['se_dfr']):>12}  "
          f"{r['dfr_strict_rate']:>8.1%}  {r['mean_asr']:>6.1%}  {r['mean_lpips']:>7.3f}")
    dfr_p.append(r['mean_dfr']); str_p.append(r['dfr_strict_rate'])
    asr_p.append(r['mean_asr']); lpips_p.append(r['mean_lpips'])
if rows_pgd:
    print(SEP)
    n_tot = sum(r['n_frames'] for r in rows_pgd)
    print(f"{'AGGREGATE':<14} {n_tot:>4}  {'':26}  "
          f"{np.mean(dfr_p):>+11.1%}   {np.mean(str_p):>8.1%}  {np.mean(asr_p):>6.1%}  {np.mean(lpips_p):>7.3f}")

# ── Markdown table for §4.18 ─────────────────────────────────────────────────
md = [
    '| Sequence | N | Classes | Method | DFR_prop | DFR_strict | ASR | LPIPS |',
    '|---|---|---|---|---|---|---|---|'
]
for seq in SELECTED:
    rc = next((r for r in rows_combined if r['seq']==seq), None)
    rp = next((r for r in rows_pgd      if r['seq']==seq), None)
    cls_str = rc['cls'] if rc else (rp['cls'] if rp else '')
    n = rc['n_frames'] if rc else (rp['n_frames'] if rp else 0)
    if rc:
        md.append(f"| {seq} | {n} | {cls_str} | Combined P3-6 "
                  f"| {rc['mean_dfr']:+.1%}\u00b1{rc['se_dfr']:.1%} "
                  f"| {rc['dfr_strict_rate']:.1%} | {rc['mean_asr']:.1%} | {rc['mean_lpips']:.3f} |")
    if rp:
        md.append(f"| {seq} | {n} | {cls_str} | PGD-mask "
                  f"| {rp['mean_dfr']:+.1%}\u00b1{rp['se_dfr']:.1%} "
                  f"| {rp['dfr_strict_rate']:.1%} | {rp['mean_asr']:.1%} | {rp['mean_lpips']:.3f} |")

if rows_combined and rows_pgd:
    n_all = sum(r['n_frames'] for r in rows_combined)
    md.append(f"| **Aggregate** | {n_all} | — | **Combined P3-6** "
              f"| **{np.mean(dfr_c):+.1%}** | **{np.mean(str_c):.1%}** "
              f"| **{np.mean(asr_c):.1%}** | **{np.mean(lpips_c):.3f}** |")
    md.append(f"| **Aggregate** | {n_all} | — | PGD-mask "
              f"| {np.mean(dfr_p):+.1%} | {np.mean(str_p):.1%} "
              f"| {np.mean(asr_p):.1%} | {np.mean(lpips_p):.3f} |")

Path('results/multiclass').mkdir(parents=True, exist_ok=True)
with open('results/multiclass/table_4_18.md', 'w') as f:
    f.write('\n'.join(md) + '\n')
print('\nSaved: results/multiclass/table_4_18.md')
print('\n' + '\n'.join(md))

## Cell M8 — Save all results to Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_PFE = '/content/drive/MyDrive/PFE'

src = Path('results/multiclass')
dst = Path(f'{DRIVE_PFE}/results/multiclass')
if src.exists():
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Copied results/multiclass → Drive')

print('\nDone. Download table_4_18.md from Drive and send the numbers.')

---
## Checklist apres le run

1. Telecharger `results/multiclass/table_4_18.md` depuis Drive
2. Envoyer les chiffres — Claude ecrit directement §4.18 dans `chapter4.tex`
3. Les sequences utilisees → noter les noms pour §4.1
4. Si DFR aggregate < 80% → expliquer dans §4.18 (scenes plus complexes)